# 07 — Multi-Distance: Breaking the (L_sd, a) Degeneracy

A single calibrant image at a single sample-detector distance
**cannot uniquely refine** both L_sd and the calibrant lattice
constant `a`.  The forward map at small 2θ is invariant under
`(L_sd, a) → (k·L_sd, k·a)` — a pure scaling.  At higher 2θ the
`tan(2θ)` and `arcsin(λ/2d)` nonlinearities break this invariance,
but slowly: a 70-ring CeO₂ pattern at 63 keV gives σ(a) of about
6 ppm at one distance, *limited by the nonlinearity-strength*, not
by detector noise.

The only fix is two (or more) calibrant images at different L_sd.
The same `a` must satisfy both images' Bragg conditions; only the
true `a` does.  This notebook walks through the analytical Fisher
information that quantifies the σ(a) collapse.


In [1]:
import os, math
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import numpy as np

# Forward model + Fisher block construction (lifted from
# dev/paper/runners/run_multi_distance.py)

def two_theta_deg(a_A, hkl_norm, lam_A):
    return 2.0 * np.degrees(np.arcsin(lam_A * hkl_norm / (2.0 * a_A)))

def fisher_block(images, *, a_truth, hkl_norm, sigma_R_px=0.05, n_per_ring=360):
    """Per-image (L_sd) + shared (a) Fisher."""
    N_im = len(images)
    n_rings = len(hkl_norm)
    rows_per_im = n_rings * n_per_ring
    n_total = N_im * rows_per_im
    J = np.zeros((n_total, N_im + 1), dtype=np.float64)
    a_truth = float(a_truth)
    for im, spec in enumerate(images):
        L_sd = spec['L_sd_um']; lam = spec['lam_A']; px = spec['px_um']
        tt = two_theta_deg(a_truth, hkl_norm, lam)
        tt_rad = np.radians(tt); th_rad = 0.5 * tt_rad
        sec2 = 1.0 / np.cos(tt_rad) ** 2
        R = L_sd * np.tan(tt_rad) / px
        col_L = np.repeat(R / L_sd, n_per_ring)
        dR_da = (L_sd / px) * sec2 * (-2.0 * np.tan(th_rad) / a_truth)
        col_a = np.repeat(dR_da, n_per_ring)
        rs = im * rows_per_im; re = rs + rows_per_im
        J[rs:re, im]    = col_L
        J[rs:re, N_im]  = col_a
    F = (J.T @ J) / (sigma_R_px ** 2)
    return F


# CeO2 70-ring set
hkl_sq = np.array([3,4,8,11,12,16,19,20,24,27,32,35,36,40,43,44,48,51,52,
                    56,59,64,67,68,72,75,76,80,83,84,88,91,96,99,100,104,107,108,
                    112,115,116,120,123,128,131,132,136,139,140,
                    144,147,148,152,155,156,160,163,164,168,171,172,176,179,180,
                    184,187,192,195,196,200], dtype=np.float64)
hkl_norm = np.sqrt(hkl_sq)
a_truth = 5.4116
lam = 0.1729
keep = (lam * hkl_norm / (2.0 * a_truth)) < 0.95
hkl_norm = hkl_norm[keep]
print(f'CeO2 rings used: {len(hkl_norm)}')


CeO2 rings used: 70


## Single-distance vs two-distance σ(a)

Build Fisher blocks for two scenarios and read off σ(a) from the
inverse Fisher.


In [2]:
def report(F, label, names):
    cov = np.linalg.pinv(F)
    sigmas = np.sqrt(np.maximum(np.diag(cov), 0.0))
    print(f'\n[{label}]')
    for n, s in zip(names, sigmas):
        unit = 'µm' if n.startswith('L') else f'Å ({s/a_truth*1e6:.2f} ppm)'
        print(f'  σ({n:<6s}) = {s:.4e}  {unit}')
    return sigmas

# Scenario A: single image at 650 mm
F_A = fisher_block(
    [dict(L_sd_um=650_000.0, lam_A=lam, px_um=172.0)],
    a_truth=a_truth, hkl_norm=hkl_norm,
)
sA = report(F_A, 'A: 1 image @ 650 mm', ['L_sd_1', 'a'])

# Scenario B: two images at 650 + 1200 mm
F_B = fisher_block(
    [dict(L_sd_um=650_000.0, lam_A=lam, px_um=172.0),
     dict(L_sd_um=1_200_000.0, lam_A=lam, px_um=172.0)],
    a_truth=a_truth, hkl_norm=hkl_norm,
)
sB = report(F_B, 'B: 2 images @ 650 + 1200 mm', ['L_sd_1', 'L_sd_2', 'a'])

print(f'\nσ(a) collapse: {sA[1]/a_truth*1e6:.2f} ppm → '
      f'{sB[-1]/a_truth*1e6:.2f} ppm  '
      f'({sA[1]/sB[-1]:.2f}× tighter)')



[A: 1 image @ 650 mm]
  σ(L_sd_1) = 4.3074e+00  µm
  σ(a     ) = 3.2153e-05  Å (5.94 ppm)

[B: 2 images @ 650 + 1200 mm]
  σ(L_sd_1) = 2.0563e+00  µm
  σ(L_sd_2) = 3.7882e+00  µm
  σ(a     ) = 1.5314e-05  Å (2.83 ppm)

σ(a) collapse: 5.94 ppm → 2.83 ppm  (2.10× tighter)


## What about adding a tight L_sd prior to the single-distance fit?

Intuition might suggest: "if I know L_sd to ±100 µm from a survey
instrument, I can pin it down and use that to refine `a`
precisely."  The intuition is **wrong**: the L_sd–a degeneracy is
not broken by an L_sd prior alone; it's only broken by an
*independent* constraint at a different L_sd.

Below: scenario A with a 100 µm L_sd prior added.  σ(a) doesn't
change.


In [3]:
F_A_prior = F_A.copy()
F_A_prior[0, 0] += 1.0 / (100.0 ** 2)    # 100 µm Gaussian prior on L_sd
sAp = report(F_A_prior, 'A + 100 µm L_sd prior', ['L_sd_1', 'a'])
print(f'\nσ(a) without/with prior: {sA[1]/a_truth*1e6:.2f} → '
      f'{sAp[1]/a_truth*1e6:.2f} ppm  (negligible change)')



[A + 100 µm L_sd prior]
  σ(L_sd_1) = 4.3034e+00  µm
  σ(a     ) = 3.2123e-05  Å (5.94 ppm)

σ(a) without/with prior: 5.94 → 5.94 ppm  (negligible change)


## Why the prior doesn't help

The (L_sd, a) Fisher block is rank-deficient at small 2θ.  Adding
a prior on L_sd adds curvature in the L_sd direction, but the
nullspace of the data Fisher *is* the joint (L_sd, a) direction —
the prior pins down L_sd, but `a` then has to follow along.
Mathematically: σ(a) ≈ σ(L_sd_prior) · ∂a/∂L_sd along the gauge
null.  At small 2θ, ∂a/∂L_sd ≈ a/L_sd ≈ 6 × 10⁻⁶ Å/µm × 100 µm =
6 × 10⁻⁴ Å ≈ 100 ppm — same scale as the data-only result.

The two-distance protocol is the only thing that adds a *new*
direction to the Fisher block (Lsd_2 is independent of Lsd_1) and
thereby breaks the gauge.

## What about adding per-ring offsets `δr_k`?

The framework also exposes per-ring radial offsets via
`add_per_ring_offset(spec, n_rings, ...)` — see notebook **05**.
These are the F2 fix for per-ring DC structure.  At a single
distance δr_k is degenerate with a per-(hkl) lattice shift Δa_k;
multi-distance breaks that degeneracy, and the framework can
recover δr_k at MAE ~0.0015 px on a 70-ring synthetic test
(see `dev/paper/runners/run_multidist_dr_k.py`).

## Practical recipe

Take two CeO₂ images at distinct L_sd at the start of every
experimental session.  Run them through the multi-image entry
point:

```python
from midas_calibrate_v2.pipelines.multi import autocalibrate_multi
res = autocalibrate_multi(
    [v1_image1, v1_image2], [image1, image2],
    spec_shared=spec_a_shared,
    specs_per_image=[spec_im1, spec_im2],
    n_iter=4, ...
)
```

The σ(a) you get back is the appropriate value to quote when
reporting absolute lattice constants downstream.
